In [1]:
import os
import re
import sys
import json
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import csv
import yaml
from collections import Counter, defaultdict

import torch
import gc

import easyocr
from ultralytics import YOLO
from transformers import AutoModelForImageTextToText, AutoProcessor
import tempfile
import time
import pytesseract

os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"



/home/cris/miniconda3/envs/vc_p4/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================================
# 1. RANDOMNESS AND TRAINING SEED
# ==========================================================
SEED = 42


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_random(seed=SEED):
    """Get a Python Random instance with a given seed. Default to global seed."""

    return random.Random(seed)


set_seed(SEED)
print(f"Training seed: {SEED}")

Training seed: 42


In [3]:
# ==========================================================
# 2. ACCELERATED DEVICE DETECTION
# ==========================================================
if torch.cuda.is_available():
    DEVICE = 0
    device_desc = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    device_desc = "Apple Silicon GPU (MPS)"
else:
    DEVICE = "cpu"
    device_desc = "CPU"

print(f"OS platform: {os.name}")
print("CPU Model: Intel i7-10700")
print(f"CPU name: {os.uname().sysname} {os.uname().release} {os.uname().version}")
print(f"CPU cores: {os.cpu_count()}")
print(f"RAM: {round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024 ** 3), 2)} GB")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"Ultralytics YOLO loaded")
print(f"Using device: {DEVICE} ({device_desc})")
print(f"VRAM available: {round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2) if torch.cuda.is_available() else 'N/A'} GB")

OS platform: posix
CPU Model: Intel i7-10700
CPU name: Linux 7.3.3-1-cachyos #1 SMP PREEMPT_DYNAMIC Sat, 07 Nov 2026 13:54:44 +0000
CPU cores: 16
RAM: 46.95 GB
Python version: 3.11.18 (main, Oct 26 2026, 18:19:35) [GCC 14.3.0]
PyTorch version: 2.6.0+cu124
OpenCV version: 4.10.0
Ultralytics YOLO loaded
Using device: 0 (NVIDIA GPU (NVIDIA GeForce RTX 3060 Ti))
VRAM available: 7.67 GB


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.cuda.is_available())

True


In [5]:
plate_detector_960_batch_16_small = YOLO(
    "../P4a/runs/detect/runs_plate/plate_detector_960_BATCH_16_small/weights/best.pt"
)
plate_detector_960_batch_16_small.to(device)

In [6]:
def clean_VRAM_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        print("VRAM cache cleared.")
    else:
        print("No GPU available to clear VRAM cache.")


def free_memory_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM libre tras limpiar: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [7]:
plate_detector_960_batch_16_small = YOLO(
    "../P4a/runs/detect/runs_plate/plate_detector_960_BATCH_16_small/weights/best.pt"
)
plate_detector_960_batch_16_small.to(device)

In [8]:
# Cargar el procesador y el modelo de SmolVLM antes de ejecutar la comparativa.
processor = AutoProcessor.from_pretrained(
    "HuggingFaceTB/SmolVLM-Instruct",
    size={"shortest_edge": 768, "longest_edge": 768},
)
model_vlm = AutoModelForImageTextToText.from_pretrained(
    "HuggingFaceTB/SmolVLM-Instruct",
    dtype=torch.bfloat16,
    _attn_implementation="eager",
    device_map="cuda"
).to(device)
model_vlm.eval()

In [9]:
import time
import re
from difflib import SequenceMatcher
import pandas as pd


def limpiar_matricula(texto):
    return re.sub(r"[^A-Z0-9]", "", str(texto).upper())


# 2. Cargar imágenes del dataset junto con sus matrículas reales de los JSON
dataset_dir = Path("../../image-dataset/prac4VC_dataset")
images_dir = dataset_dir / "images"
annotations_dir = dataset_dir / "annotations"

muestras_dataset = []
for img_path in sorted(images_dir.iterdir()):
    if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue
    json_path = annotations_dir / f"{img_path.stem}.json"
    if not json_path.exists():
        continue

    with open(json_path, "r", encoding="utf-8") as f:
        anotaciones = json.load(f)

    # Extraer matrículas reales válidas (ignorando vacías o placeholders '0000AAA')
    placas_reales = [
        limpiar_matricula(a.get("plate", ""))
        for a in anotaciones
        if limpiar_matricula(a.get("plate", "")) not in ["", "0000AAA"]
    ]

    if len(placas_reales) > 0:
        muestras_dataset.append((img_path, placas_reales))

# Seleccionar un subconjunto fijo de imágenes para evaluar en las mismas condiciones
rdm = get_random(SEED)
MAX_IMAGENES = 50  # Puedes cambiarlo (ej. 30, 50, 100)
muestras_eval = rdm.sample(
    muestras_dataset, min(MAX_IMAGENES, len(muestras_dataset))
)

print(
    f"Total imágenes válidas: {len(muestras_dataset)} | Seleccionadas para evaluar: {len(muestras_eval)}"
)

Total imágenes válidas: 2387 | Seleccionadas para evaluar: 50


In [10]:
import gc
import time
from difflib import SequenceMatcher
import pandas as pd


PLATE_ALLOWLIST = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"


def preprocesar_recorte(img, box, pad=5):
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
    h, w = img.shape[:2]
    recorte = img[
        max(0, int(y1) - pad) : min(h, int(y2) + pad),
        max(0, int(x1) - pad) : min(w, int(x2) + pad),
    ].copy()

    rh, rw = recorte.shape[:2]
    if rh < 100 or rw < 300:
        scale = min(3.0, 600.0 / max(rw, 1))
        if scale > 1.0:
            recorte = cv2.resize(
                recorte, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC
            )

    recorte_gris = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)
    return cv2.GaussianBlur(recorte_gris, (5, 5), 0)


def leer_con_easyocr(recorte_gris, reader):
    t0 = time.perf_counter()
    textos = reader.readtext(recorte_gris, detail=0, allowlist=PLATE_ALLOWLIST)
    t_inferencia = time.perf_counter() - t0

    candidato = limpiar_matricula("".join(textos))
    for texto in textos:
        texto_limpio = limpiar_matricula(texto)
        if len(texto_limpio) == 7:
            candidato = texto_limpio
            break
    return candidato, t_inferencia


def leer_con_tesseract(recorte_gris):
    idiomas = pytesseract.get_languages(config="")
    if "spa" not in idiomas:
        raise RuntimeError(
            "Tesseract necesita el paquete de idioma 'spa'. "
            f"Idiomas disponibles: {idiomas}"
        )

    t0 = time.perf_counter()
    texto = pytesseract.image_to_string(
        recorte_gris,
        lang="spa",
        config="--psm 7 -c tessedit_char_whitelist=" + PLATE_ALLOWLIST,
    )
    t_inferencia = time.perf_counter() - t0
    return limpiar_matricula(texto), t_inferencia


def leer_con_unlimited_ocr(recorte_gris, model, tokenizer):
    # Unlimited-OCR recibe una ruta de imagen y escribe el resultado en result.md.
    t0 = time.perf_counter()
    with tempfile.TemporaryDirectory(prefix="unlimited_ocr_") as temp_dir:
        image_path = Path(temp_dir) / "plate.png"
        output_path = Path(temp_dir) / "output"
        cv2.imwrite(str(image_path), recorte_gris)
        model.infer(
            tokenizer,
            prompt="<image>document parsing.",
            image_file=str(image_path),
            output_path=str(output_path),
            base_size=1024,
            image_size=640,
            crop_mode=True,
            max_length=128,
            no_repeat_ngram_size=35,
            ngram_window=128,
            save_results=True,
        )
        result_path = output_path / "result.md"
        if not result_path.exists():
            raise RuntimeError(f"Unlimited-OCR no generó {result_path}")
        texto = result_path.read_text(encoding="utf-8")

    t_inferencia = time.perf_counter() - t0
    return limpiar_matricula(texto), t_inferencia


def leer_con_smolvlm(recorte_gris):
    t0 = time.perf_counter()
    pil_img = Image.fromarray(cv2.cvtColor(recorte_gris, cv2.COLOR_GRAY2RGB))
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": "Read the license plate text in this image. Reply ONLY with the alphanumeric characters of the plate, without spaces or hyphens.",
                },
            ],
        }
    ]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[pil_img], return_tensors="pt").to(device)

    with torch.inference_mode():
        generated_ids = model_vlm.generate(
            **inputs, max_new_tokens=10, do_sample=False
        )
        generated_text = processor.batch_decode(
            generated_ids[:, inputs["input_ids"].shape[1] :],
            skip_special_tokens=True,
        )[0]

    del inputs, generated_ids
    return limpiar_matricula(generated_text), time.perf_counter() - t0


def calcular_metricas_ocr(prediccion, placas_reales):
    if not prediccion:
        return 0, 0.0, ""
    mejor_real = max(
        placas_reales, key=lambda r: SequenceMatcher(None, prediccion, r).ratio()
    )
    similitud = SequenceMatcher(None, prediccion, mejor_real).ratio()
    acierto_exacto = int(prediccion == mejor_real)
    return acierto_exacto, similitud, mejor_real


In [11]:
# La detección se realiza una sola vez para que todos los OCR reciban los mismos recortes.
resultados_comparativa = []
recortes_detectados = []

print("1/4 Detectando matrículas con YOLO...")
for img_path, placas_reales in muestras_eval:
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    t0_yolo = time.perf_counter()
    preds_yolo = plate_detector_960_batch_16_small.predict(
        source=img, conf=0.5, verbose=False, device=device
    )
    t_yolo = time.perf_counter() - t0_yolo
    cajas = preds_yolo[0].boxes

    if len(cajas) == 0:
        recortes_detectados.append((img_path.name, placas_reales, None, t_yolo))
    else:
        for box in cajas:
            recortes_detectados.append(
                (img_path.name, placas_reales, preprocesar_recorte(img, box), t_yolo)
            )




def guardar_resultado(modelo, idx_recorte, img_name, placas_reales, recorte, t_yolo, lector):
    if recorte is None:
        prediccion, t_ocr = "", 0.0
        real = placas_reales[0]
        exacto, similitud = 0, 0.0
    else:
        prediccion, t_ocr = lector(recorte)
        exacto, similitud, real = calcular_metricas_ocr(prediccion, placas_reales)
        real = real or placas_reales[0]

    resultados_comparativa.append({
        "id_recorte": idx_recorte,
        "imagen": img_name,
        "modelo_ocr": modelo,
        "matricula_real": real,
        "prediccion": prediccion,
        "acierto_exacto": exacto,
        "similitud_caracteres": similitud,
        "tiempo_ocr_s": t_ocr,
        "tiempo_total_s": t_yolo + t_ocr,
    })


def ejecutar_backend(modelo, lector):
    print(f"Evaluando {modelo} sobre {len(recortes_detectados)} recortes...")
    for idx, (img_name, placas_reales, recorte, t_yolo) in enumerate(recortes_detectados):
        guardar_resultado(modelo, idx, img_name, placas_reales, recorte, t_yolo, lector)


def guardar_resultados_parciales():
    Path("results").mkdir(exist_ok=True)
    pd.DataFrame(resultados_comparativa).to_csv(
        "results/comparativa_resultados_parciales.csv", index=False
    )

1/4 Detectando matrículas con YOLO...


In [12]:
# Ejecuta esta celda después de la detección y antes de cada backend.
if "resultados_comparativa" not in globals():
    resultados_comparativa = []

# EasyOCR
if "model_vlm" in globals():
    model_vlm.to("cpu")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
reader_easy = easyocr.Reader(["es"], gpu=torch.cuda.is_available(), verbose=False)
ejecutar_backend("EasyOCR", lambda recorte: leer_con_easyocr(recorte, reader_easy))
free_memory_model(reader_easy)
guardar_resultados_parciales()

Evaluando EasyOCR sobre 79 recortes...
VRAM libre tras limpiar: 0.23 GB


In [14]:
# PaddleOCR aislado en un proceso independiente.
# Así PaddlePaddle no comparte el kernel con PyTorch/EasyOCR/SmolVLM.
import subprocess
import tempfile

worker = Path("paddle_ocr_worker.py").resolve()
if not worker.exists():
    raise FileNotFoundError(f"No se encuentra el worker: {worker}")

with tempfile.TemporaryDirectory(prefix="paddle_recortes_") as temp_dir:
    temp_dir = Path(temp_dir)
    salida_json = temp_dir / "paddle_resultados.json"
    recortes_paddle = {}

    for idx, (img_name, placas_reales, recorte, t_yolo) in enumerate(recortes_detectados):
        if recorte is None:
            continue
        ruta_recorte = temp_dir / f"{idx}.png"
        if not cv2.imwrite(str(ruta_recorte), recorte):
            raise RuntimeError(f"No se pudo guardar el recorte {idx}")
        recortes_paddle[idx] = (img_name, placas_reales, t_yolo)

    proceso = subprocess.run(
        [sys.executable, str(worker), str(temp_dir), str(salida_json), "--lang", "es"],
        check=True,
        text=True,
        capture_output=True,
    )
    if proceso.stderr:
        print(proceso.stderr.strip())

    resultados_paddle = json.loads(salida_json.read_text(encoding="utf-8"))

for idx, (img_name, placas_reales, recorte, t_yolo) in enumerate(recortes_detectados):
    if recorte is None:
        prediccion, t_ocr = "", 0.0
        real = placas_reales[0]
        exacto, similitud = 0, 0.0
    else:
        datos = resultados_paddle.get(str(idx), {})
        prediccion = limpiar_matricula("".join(datos.get("textos", [])))
        t_ocr = float(datos.get("tiempo_ocr_s", 0.0))
        exacto, similitud, real = calcular_metricas_ocr(prediccion, placas_reales)
        real = real or placas_reales[0]

    resultados_comparativa.append({
        "id_recorte": idx,
        "imagen": img_name,
        "modelo_ocr": "PaddleOCR",
        "matricula_real": real,
        "prediccion": prediccion,
        "acierto_exacto": exacto,
        "similitud_caracteres": similitud,
        "tiempo_ocr_s": t_ocr,
        "tiempo_total_s": t_yolo + t_ocr,
    })

guardar_resultados_parciales()
print(f"PaddleOCR integrado: {len(recortes_detectados)} recortes.")


PaddleOCR integrado: 79 recortes.


In [15]:
# Ejecuta esta celda después de la detección y antes de cada backend.
if "resultados_comparativa" not in globals():
    resultados_comparativa = []

# SmolVLM
if "model_vlm" not in globals():
    raise RuntimeError("Ejecuta primero la celda de carga de SmolVLM.")
model_vlm.to(device)
ejecutar_backend("SmolVLM", leer_con_smolvlm)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
guardar_resultados_parciales()

Evaluando SmolVLM sobre 79 recortes...


In [16]:
# Resumen de los backends ejecutados.
df_resultados = pd.DataFrame(resultados_comparativa)
resumen = (
    df_resultados.groupby("modelo_ocr", sort=False)
    .agg(
        matriculas_evaluadas=("imagen", "count"),
        tasa_acierto_exacto_pct=("acierto_exacto", lambda x: round(x.mean() * 100, 2)),
        similitud_media_pct=("similitud_caracteres", lambda x: round(x.mean() * 100, 2)),
        tiempo_medio_ocr_ms=("tiempo_ocr_s", lambda x: round(x.mean() * 1000, 2)),
        tiempo_medio_total_ms=("tiempo_total_s", lambda x: round(x.mean() * 1000, 2)),
    )
    .reset_index()
)
display(resumen)

In [17]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
resumen.plot.bar(
    x="modelo_ocr", y="tasa_acierto_exacto_pct", ax=axes[0], legend=False,
    color="#4c78a8", title="Acierto exacto (%)"
)
resumen.plot.bar(
    x="modelo_ocr", y="tiempo_medio_total_ms", ax=axes[1], legend=False,
    color="#f58518", title="Tiempo total medio (ms)"
)
axes[0].set_xlabel("")
axes[1].set_xlabel("")
axes[0].set_ylabel("Porcentaje")
axes[1].set_ylabel("Milisegundos")
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
